In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install torch numpy scikit-learn

"""
Assamese Spell Checker - EXACT SPECIFICATION
1. Dictionary from raw .txt file
2. Dictionary lookup → CNN embeddings → Agglomerative clustering (Euclidean)
3. Top-3 cluster suggestions + Best cluster identification
"""

In [ ]:
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import euclidean_distances
from collections import Counter
import io

#CNN+Cosine+Time

In [ ]:
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from collections import defaultdict, Counter
print("="*80)
class CNNEmbedder(nn.Module):
    def __init__(self, vocab_size=128, embed_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, 64, k, padding=k//2) for k in [2, 3, 4]
        ])
        self.fc = nn.Linear(192, 128)

    def forward(self, x):
        seq_len = x.size(1)
        if seq_len < 12:
            x = F.pad(x, (0, 12 - seq_len))
        x = self.embedding(x).transpose(1, 2)
        conv_outs = [F.relu(conv(x)) for conv in self.convs]
        pooled = [F.adaptive_max_pool1d(out, 1).squeeze(-1) for out in conv_outs]
        return F.normalize(self.fc(torch.cat(pooled, 1)), dim=1)
class PureCosineSpellChecker:
    def __init__(self, raw_file_path):
        self.dictionary = self._load_dictionary(raw_file_path)
        print(f"Dictionary: {len(self.dictionary):,} words")

        self.cnn_model = CNNEmbedder()
        self._train_cnn_fast()
        self.embedding_cache = {}

        self.word_embeddings = self._precompute_embeddings()
        self.length_index = self._build_length_index()
        print(f"CNN Embeddings")

    def _load_dictionary(self, file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
        except:
            text = "অসম ভাৰতৰ উত্তৰ-পূবত অৱস্থিত।"
        words = re.findall(r'[\u0980-\u09FF]{2,12}', text)
        return sorted(list(set(words)))

    def _train_cnn_fast(self):
        print("CNN Training...")
        model = self.cnn_model
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

        words_sample = self.dictionary[:400]
        if len(words_sample) < 2:
            return

        indices_batch = []
        for word in words_sample:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long)
            indices_batch.append(indices)

        indices_tensor = torch.stack(indices_batch)

        model.train()
        for epoch in range(10):
            embeddings = model(indices_tensor)
            sim_matrix = torch.mm(embeddings, embeddings.t()) / 0.5
            labels = torch.arange(len(words_sample))
            mask = torch.eye(len(words_sample)).bool()
            sim_matrix.masked_fill_(mask, -9e15)
            loss = F.cross_entropy(sim_matrix, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()

    def _precompute_embeddings(self):
        embeddings = {}
        for i, word in enumerate(self.dictionary):
            if i % 2000 == 0:
                print(f"  Precomputing: {i}/{len(self.dictionary)}")
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                embeddings[word] = self.cnn_model(indices).cpu().numpy()[0]
        return embeddings

    def _build_length_index(self):
        index = defaultdict(list)
        for word in self.dictionary:
            index[len(word)].append(word)
        return dict(index)

    def get_embedding(self, word):
        if word not in self.embedding_cache:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                self.embedding_cache[word] = self.cnn_model(indices).cpu().numpy()[0]
        return self.embedding_cache[word]

    def COSINE_similarity(self, word1, word2):
        emb1 = self.get_embedding(word1)
        emb2 = self.word_embeddings[word2]
        return np.dot(emb1, emb2)

    def check(self, word):
        if len(word) < 2 or word in self.dictionary:
            return word in self.dictionary, []

        word_len = len(word)
        candidates = []
        for len_diff in [-1, 0, 1]:
            target_len = word_len + len_diff
            if target_len in self.length_index:
                candidates.extend(self.length_index[target_len][:150])
        candidates = list(set(candidates))[:350]
        scored = [(self.COSINE_similarity(word, cand), cand) for cand in candidates]
        top5 = sorted(scored, reverse=True)[:5]
        return False, top5
def generate_test_cases(dictionary, n_samples=200):
    error_swaps = {'া': 'ি', 'ি': 'া', 'ক': 'খ', 'খ': 'ক', 'গ': 'ঘ'}
    test_cases = []
    valid_words = [w for w in dictionary if 3 <= len(w) <= 10]

    for word in valid_words:
        for wrong, right in error_swaps.items():
            if wrong in word:
                misspelled = word.replace(wrong, right, 1)
                if misspelled != word and len(misspelled) >= 2:
                    test_cases.append({'original': word, 'misspelled': misspelled})
                    if len(test_cases) >= n_samples:
                        break
        if len(test_cases) >= n_samples:
            break
    print(f"Generated {len(test_cases)} test cases")
    return test_cases
def evaluate_pure_cosine(checker, test_cases):
    print("\nCOSINE EVALUATION")
    print("="*60)

    top1_correct = top3_correct = top5_correct = 0

    #TIMER
    start_time = time.time()

    print("Sample corrections:")
    for i, case in enumerate(test_cases[:10]):
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()

        top1_ok = case['original'] == top1_word
        top3_ok = case['original'] in top3_words

        status = 'OK' if top3_ok else 'ERROR'
        print(f"  '{case['misspelled']}' → {top1_word} | {case['original']} | {status}")

        top1_correct += int(top1_ok)
        top3_correct += int(top3_ok)
        top5_correct += int(case['original'] in {s[1] for s in suggestions[:5]})

    #EVALUATION
    for case in test_cases[10:]:
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()
        top1_correct += int(case['original'] == top1_word)
        top3_correct += int(case['original'] in top3_words)
        top5_correct += int(case['original'] in {s[1] for s in suggestions[:5]})

    #TIMER
    end_time = time.time()
    total_duration = end_time - start_time

    total = len(test_cases)
    print(f"\n{'='*60}")
    print(f"COSINE SIMILARITY")
    print(f"Dictionary: {len(checker.dictionary):,} words")
    print(f"Test cases: {total}")
    print(f"Execution Time: {total_duration:.4f} seconds")
    print(f"Speed: {total_duration/total:.4f} sec/word")
    print(f"Top-1 Accuracy: {top1_correct/total:.1%}")
    print(f"Top-3 Accuracy: {top3_correct/total:.1%}")
    print(f"Top-5 Accuracy: {top5_correct/total:.1%}")
    print(f"{'='*60}")


    print(f"Cosine: {top3_correct/total:.1%} ")

#MAIN
if __name__ == "__main__":
    raw_file = "News.txt"
    checker = PureCosineSpellChecker(raw_file)

    test_cases = generate_test_cases(checker.dictionary, 1000)
    evaluate_pure_cosine(checker, test_cases)

Dictionary: 56,544 words
🔄 CNN Training...
  Precomputing: 0/56544
  Precomputing: 2000/56544
  Precomputing: 4000/56544
  Precomputing: 6000/56544
  Precomputing: 8000/56544
  Precomputing: 10000/56544
  Precomputing: 12000/56544
  Precomputing: 14000/56544
  Precomputing: 16000/56544
  Precomputing: 18000/56544
  Precomputing: 20000/56544
  Precomputing: 22000/56544
  Precomputing: 24000/56544
  Precomputing: 26000/56544
  Precomputing: 28000/56544
  Precomputing: 30000/56544
  Precomputing: 32000/56544
  Precomputing: 34000/56544
  Precomputing: 36000/56544
  Precomputing: 38000/56544
  Precomputing: 40000/56544
  Precomputing: 42000/56544
  Precomputing: 44000/56544
  Precomputing: 46000/56544
  Precomputing: 48000/56544
  Precomputing: 50000/56544
  Precomputing: 52000/56544
  Precomputing: 54000/56544
  Precomputing: 56000/56544
Pure Cosine Ready! (CNN Embeddings Only)
Generated 1000 test cases

🎯 PURE COSINE EVALUATION
Sample corrections:
  'ংখ্ৰমণে' → ংক্ৰমণে | ংক্ৰমণে | ✅
  'অ

#CNN + Levenshtien+Time

In [ ]:
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from collections import defaultdict, Counter

print(" CNN + LEVENSHTEIN DISTANCE ")
print("="*80)

class CNNEmbedder(nn.Module):
    def __init__(self, vocab_size=128, embed_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, 64, k, padding=k//2) for k in [2, 3, 4]
        ])
        self.fc = nn.Linear(192, 128)

    def forward(self, x):
        seq_len = x.size(1)
        if seq_len < 12:
            x = F.pad(x, (0, 12 - seq_len))
        x = self.embedding(x).transpose(1, 2)
        conv_outs = [F.relu(conv(x)) for conv in self.convs]
        pooled = [F.adaptive_max_pool1d(out, 1).squeeze(-1) for out in conv_outs]
        return F.normalize(self.fc(torch.cat(pooled, 1)), dim=1)
class LevenshteinCNNSpellChecker:
    def __init__(self, raw_file_path):
        print(" CNN + Levenshtein Initializing...")

        self.dictionary = self._load_dictionary(raw_file_path)
        print(f" Dictionary: {len(self.dictionary):,} words")

        self.cnn_model = CNNEmbedder()
        self._train_cnn_fast()
        self.embedding_cache = {}

        self.word_embeddings = self._precompute_embeddings()
        self.length_index = self._build_length_index()
        print(f" CNN + Levenshtein Ready!")

    def _load_dictionary(self, file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
        except:
            text = "অসম ভাৰতৰ উত্তৰ-পূবত অৱস্থিত।"
        words = re.findall(r'[\u0980-\u09FF]{2,12}', text)
        return sorted(list(set(words)))

    def _train_cnn_fast(self):
        print(" CNN Training...")
        model = self.cnn_model
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

        words_sample = self.dictionary[:400]
        if len(words_sample) < 2:
            return

        indices_batch = []
        for word in words_sample:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long)
            indices_batch.append(indices)

        indices_tensor = torch.stack(indices_batch)

        model.train()
        for epoch in range(10):
            embeddings = model(indices_tensor)
            sim_matrix = torch.mm(embeddings, embeddings.t()) / 0.5
            labels = torch.arange(len(words_sample))
            mask = torch.eye(len(words_sample)).bool()
            sim_matrix.masked_fill_(mask, -9e15)
            loss = F.cross_entropy(sim_matrix, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()

    def _precompute_embeddings(self):
        embeddings = {}
        for i, word in enumerate(self.dictionary):
            if i % 2000 == 0:
                print(f"  Precomputing: {i}/{len(self.dictionary)}")
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                embeddings[word] = self.cnn_model(indices).cpu().numpy()[0]
        return embeddings

    def _build_length_index(self):
        index = defaultdict(list)
        for word in self.dictionary:
            index[len(word)].append(word)
        return dict(index)

    def get_embedding(self, word):
        if word not in self.embedding_cache:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                self.embedding_cache[word] = self.cnn_model(indices).cpu().numpy()[0]
        return self.embedding_cache[word]

    def levenshtein_distance(self, s1, s2):
        if len(s1) < len(s2):
            return self.levenshtein_distance(s2, s1)

        if len(s2) == 0:
            return len(s1)

        previous_row = range(len(s2) + 1)
        for i, c1 in enumerate(s1):
            current_row = [i + 1]
            for j, c2 in enumerate(s2):
                insertions = previous_row[j + 1] + 1
                deletions = current_row[j] + 1
                substitutions = previous_row[j] + (c1 != c2)
                current_row.append(min(insertions, deletions, substitutions))
            previous_row = current_row
        return previous_row[-1] / max(len(s1), len(s2), 1)

    def check(self, word):
        if len(word) < 2 or word in self.dictionary:
            return word in self.dictionary, []

        word_len = len(word)
        candidates = []
        for len_diff in [-1, 0, 1, 2]:
            target_len = word_len + len_diff
            if target_len in self.length_index:
                candidates.extend(self.length_index[target_len][:120])
        candidates = list(set(candidates))[:400]
        scored = [(-self.levenshtein_distance(word, cand), cand) for cand in candidates]
        top5 = sorted(scored, reverse=True)[:5]
        return False, top5
def generate_test_cases(dictionary, n_samples=200):
    error_swaps = {'া': 'ি', 'ি': 'া', 'ক': 'খ', 'খ': 'ক', 'গ': 'ঘ'}
    test_cases = []
    valid_words = [w for w in dictionary if 3 <= len(w) <= 10]

    for word in valid_words:
        for wrong, right in error_swaps.items():
            if wrong in word:
                misspelled = word.replace(wrong, right, 1)
                if misspelled != word and len(misspelled) >= 2:
                    test_cases.append({'original': word, 'misspelled': misspelled})
                    if len(test_cases) >= n_samples:
                        break
        if len(test_cases) >= n_samples:
            break
    print(f" Generated {len(test_cases)} test cases")
    return test_cases
def evaluate_levenshtein_cnn(checker, test_cases):
    print("\n CNN + LEVENSHTEIN EVALUATION")
    print("="*60)

    top1_correct = top3_correct = top5_correct = 0

    #TIMER
    start_time = time.time()

    print("Sample corrections:")
    for i, case in enumerate(test_cases[:10]):
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()
        top5_words = {s[1] for s in suggestions[:5]} if len(suggestions) >= 5 else set()

        top1_ok = case['original'] == top1_word
        top3_ok = case['original'] in top3_words
        top5_ok = case['original'] in top5_words

        status = 'OK' if top3_ok else 'ERROR'
        print(f"  '{case['misspelled']}' → {top1_word} | {case['original']} | {status}")

        top1_correct += int(top1_ok)
        top3_correct += int(top3_ok)
        top5_correct += int(top5_ok)

    #EVALUATION
    for case in test_cases[10:]:
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()
        top5_words = {s[1] for s in suggestions[:5]} if len(suggestions) >= 5 else set()

        top1_correct += int(case['original'] == top1_word)
        top3_correct += int(case['original'] in top3_words)
        top5_correct += int(case['original'] in top5_words)

    #TIMER
    end_time = time.time()
    total_duration = end_time - start_time

    total = len(test_cases)
    print(f"\n{'='*60}")
    print(f" CNN + LEVENSHTEIN DISTANCE")
    print(f" Dictionary: {len(checker.dictionary):,} words")
    print(f" Test cases: {total}")
    print(f" Execution Time: {total_duration:.4f} seconds")
    print(f" Speed: {total_duration/total:.4f} sec/word")
    print(f" Top-1 Accuracy: {top1_correct/total:.1%}")
    print(f" Top-3 Accuracy: {top3_correct/total:.1%}")
    print(f"  Top-5 Accuracy: {top5_correct/total:.1%}")
    print(f"{'='*60}")

    print(f"\n COMPARISON:")
    print(f"CNN + Levenshtein: **{top3_correct/total:.1%}** ")

#MAIN
if __name__ == "__main__":
    raw_file = "News.txt"
    checker = LevenshteinCNNSpellChecker(raw_file)

    test_cases = generate_test_cases(checker.dictionary, 1000)
    evaluate_levenshtein_cnn(checker, test_cases)

 CNN + LEVENSHTEIN DISTANCE 
 CNN + Levenshtein Initializing...
 Dictionary: 56,544 words
 CNN Training...
  Precomputing: 0/56544
  Precomputing: 2000/56544
  Precomputing: 4000/56544
  Precomputing: 6000/56544
  Precomputing: 8000/56544
  Precomputing: 10000/56544
  Precomputing: 12000/56544
  Precomputing: 14000/56544
  Precomputing: 16000/56544
  Precomputing: 18000/56544
  Precomputing: 20000/56544
  Precomputing: 22000/56544
  Precomputing: 24000/56544
  Precomputing: 26000/56544
  Precomputing: 28000/56544
  Precomputing: 30000/56544
  Precomputing: 32000/56544
  Precomputing: 34000/56544
  Precomputing: 36000/56544
  Precomputing: 38000/56544
  Precomputing: 40000/56544
  Precomputing: 42000/56544
  Precomputing: 44000/56544
  Precomputing: 46000/56544
  Precomputing: 48000/56544
  Precomputing: 50000/56544
  Precomputing: 52000/56544
  Precomputing: 54000/56544
  Precomputing: 56000/56544
 CNN + Levenshtein Ready!
 Generated 1000 test cases

 CNN + LEVENSHTEIN EVALUATION
🔍 Sam

#CNN+Jaro+Time

In [ ]:
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from collections import defaultdict, Counter

print(" CNN + JARO-WINKLER DISTANCE ")
print("="*80)
def jaro_winkler_similarity(s1, s2):
    if len(s1) == 0 and len(s2) == 0:
        return 1.0
    if len(s1) == 0 or len(s2) == 0:
        return 0.0

    match_window = max(len(s1), len(s2)) // 2 - 1
    if match_window < 0:
        match_window = 0

    hash_s1 = [False] * len(s1)
    hash_s2 = [False] * len(s2)
    matches1, matches2 = 0, 0
    for i, c1 in enumerate(s1):
        start = max(0, i - match_window)
        end = min(i + match_window + 1, len(s2))
        for j in range(start, end):
            if c1 == s2[j] and not hash_s2[j]:
                hash_s1[i] = hash_s2[j] = True
                matches1 += 1
                break
    for i, c2 in enumerate(s2):
        start = max(0, i - match_window)
        end = min(i + match_window + 1, len(s1))
        for j in range(start, end):
            if c2 == s1[j] and not hash_s1[j]:
                hash_s2[i] = hash_s1[j] = True
                matches2 += 1
                break

    m = min(matches1, matches2)

    if m == 0:
        return 0.0
    t = 0
    for i in range(len(s1)):
        if hash_s1[i]:
            j = next((k for k in range(len(s2)) if hash_s2[k] and s1[i] == s2[k]), None)
            if j and j != i:
                t += 1
    t = t // 2
    jaro = (m / len(s1) + m / len(s2) + (m - t) / m) / 3.0
    prefix = 0
    for i in range(min(len(s1), len(s2), 4)):
        if s1[i] == s2[i]:
            prefix += 1
        else:
            break

    p = 0.1
    return min(1.0, jaro + prefix * p * (1.0 - jaro))

#CNN MODEL
class CNNEmbedder(nn.Module):
    def __init__(self, vocab_size=128, embed_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, 64, k, padding=k//2) for k in [2, 3, 4]
        ])
        self.fc = nn.Linear(192, 128)

    def forward(self, x):
        seq_len = x.size(1)
        if seq_len < 12:
            x = F.pad(x, (0, 12 - seq_len))
        x = self.embedding(x).transpose(1, 2)
        conv_outs = [F.relu(conv(x)) for conv in self.convs]
        pooled = [F.adaptive_max_pool1d(out, 1).squeeze(-1) for out in conv_outs]
        return F.normalize(self.fc(torch.cat(pooled, 1)), dim=1)
class JaroWinklerCNNSpellChecker:
    def __init__(self, raw_file_path):
        print(" CNN + Jaro-Winkler Initializing...")

        self.dictionary = self._load_dictionary(raw_file_path)
        print(f" Dictionary: {len(self.dictionary):,} words")

        self.cnn_model = CNNEmbedder()
        self._train_cnn_fast()
        self.embedding_cache = {}

        self.word_embeddings = self._precompute_embeddings()
        self.length_index = self._build_length_index()
        print(f" CNN + Jaro-Winkler Ready!")

    def _load_dictionary(self, file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
        except:
            text = "অসম ভাৰতৰ উত্তৰ-পূবত অৱস্থিত।"
        words = re.findall(r'[\u0980-\u09FF]{2,12}', text)
        return sorted(list(set(words)))

    def _train_cnn_fast(self):
        print(" CNN Training...")
        model = self.cnn_model
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

        words_sample = self.dictionary[:400]
        if len(words_sample) < 2:
            return

        indices_batch = []
        for word in words_sample:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long)
            indices_batch.append(indices)

        indices_tensor = torch.stack(indices_batch)

        model.train()
        for epoch in range(10):
            embeddings = model(indices_tensor)
            sim_matrix = torch.mm(embeddings, embeddings.t()) / 0.5
            labels = torch.arange(len(words_sample))
            mask = torch.eye(len(words_sample)).bool()
            sim_matrix.masked_fill_(mask, -9e15)
            loss = F.cross_entropy(sim_matrix, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()

    def _precompute_embeddings(self):
        embeddings = {}
        for i, word in enumerate(self.dictionary):
            if i % 2000 == 0:
                print(f"  Precomputing: {i}/{len(self.dictionary)}")
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                embeddings[word] = self.cnn_model(indices).cpu().numpy()[0]
        return embeddings

    def _build_length_index(self):
        index = defaultdict(list)
        for word in self.dictionary:
            index[len(word)].append(word)
        return dict(index)

    def get_embedding(self, word):
        if word not in self.embedding_cache:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                self.embedding_cache[word] = self.cnn_model(indices).cpu().numpy()[0]
        return self.embedding_cache[word]

    def JARO_WINKLER_similarity(self, word1, word2):
        return jaro_winkler_similarity(word1, word2)

    def check(self, word):
        if len(word) < 2 or word in self.dictionary:
            return word in self.dictionary, []

        word_len = len(word)
        candidates = []
        for len_diff in [-1, 0, 1]:
            target_len = word_len + len_diff
            if target_len in self.length_index:
                candidates.extend(self.length_index[target_len][:150])

        candidates = list(set(candidates))[:350]
        scored = [(self.JARO_WINKLER_similarity(word, cand), cand) for cand in candidates]
        top5 = sorted(scored, reverse=True)[:5]

        return False, top5
def generate_test_cases(dictionary, n_samples=200):
    error_swaps = {'া': 'ি', 'ি': 'া', 'ক': 'খ', 'খ': 'ক', 'গ': 'ঘ'}
    test_cases = []
    valid_words = [w for w in dictionary if 3 <= len(w) <= 10]

    for word in valid_words:
        for wrong, right in error_swaps.items():
            if wrong in word:
                misspelled = word.replace(wrong, right, 1)
                if misspelled != word and len(misspelled) >= 2:
                    test_cases.append({'original': word, 'misspelled': misspelled})
                    if len(test_cases) >= n_samples:
                        break
        if len(test_cases) >= n_samples:
            break
    print(f" Generated {len(test_cases)} test cases")
    return test_cases

#EVALUATION
def evaluate_jaro_cnn(checker, test_cases):
    print("\n CNN + JARO-WINKLER EVALUATION")
    print("="*60)

    top1_correct = top3_correct = top5_correct = 0

    #TIMER
    start_time = time.time()

    print("Sample corrections:")
    for i, case in enumerate(test_cases[:10]):
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()
        top5_words = {s[1] for s in suggestions[:5]} if len(suggestions) >= 5 else set()

        top1_ok = case['original'] == top1_word
        top3_ok = case['original'] in top3_words
        top5_ok = case['original'] in top5_words

        status = 'OK' if top3_ok else 'ERROR'
        print(f"  '{case['misspelled']}' → {top1_word} | {case['original']} | {status}")

        top1_correct += int(top1_ok)
        top3_correct += int(top3_ok)
        top5_correct += int(top5_ok)

    #EVALUATION
    for case in test_cases[10:]:
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()
        top5_words = {s[1] for s in suggestions[:5]} if len(suggestions) >= 5 else set()

        top1_correct += int(case['original'] == top1_word)
        top3_correct += int(case['original'] in top3_words)
        top5_correct += int(case['original'] in top5_words)

    #TIMER
    end_time = time.time()
    total_duration = end_time - start_time

    total = len(test_cases)
    print(f"\n{'='*60}")
    print(f" CNN + JARO-WINKLER DISTANCE")
    print(f" Dictionary: {len(checker.dictionary):,} words")
    print(f" Test cases: {total}")
    print(f" Execution Time: {total_duration:.4f} seconds")
    print(f" Speed: {total_duration/total:.4f} sec/word")
    print(f" Top-1 Accuracy: {top1_correct/total:.1%}")
    print(f" Top-3 Accuracy: {top3_correct/total:.1%}")
    print(f" Top-5 Accuracy: {top5_correct/total:.1%}")
    print(f"{'='*60}")

    print(f"\nCOMPARISON:")
    print(f"CNN + Jaro-Winkler: **{top3_correct/total:.1%}** ")

# ---------- RUN ----------
if __name__ == "__main__":
    raw_file = "News.txt"
    checker = JaroWinklerCNNSpellChecker(raw_file)

    test_cases = generate_test_cases(checker.dictionary, 1000)
    evaluate_jaro_cnn(checker, test_cases)

 CNN + JARO-WINKLER DISTANCE 
 CNN + Jaro-Winkler Initializing...
 Dictionary: 56,544 words
 CNN Training...
  Precomputing: 0/56544
  Precomputing: 2000/56544
  Precomputing: 4000/56544
  Precomputing: 6000/56544
  Precomputing: 8000/56544
  Precomputing: 10000/56544
  Precomputing: 12000/56544
  Precomputing: 14000/56544
  Precomputing: 16000/56544
  Precomputing: 18000/56544
  Precomputing: 20000/56544
  Precomputing: 22000/56544
  Precomputing: 24000/56544
  Precomputing: 26000/56544
  Precomputing: 28000/56544
  Precomputing: 30000/56544
  Precomputing: 32000/56544
  Precomputing: 34000/56544
  Precomputing: 36000/56544
  Precomputing: 38000/56544
  Precomputing: 40000/56544
  Precomputing: 42000/56544
  Precomputing: 44000/56544
  Precomputing: 46000/56544
  Precomputing: 48000/56544
  Precomputing: 50000/56544
  Precomputing: 52000/56544
  Precomputing: 54000/56544
  Precomputing: 56000/56544
 CNN + Jaro-Winkler Ready!
 Generated 1000 test cases

 CNN + JARO-WINKLER EVALUATION
🔍

#CNN+ Edit + Jaro+ Time

In [ ]:
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from collections import defaultdict, Counter

print("CNN + Edit + Jaro-Winkler")
print("="*80)

def jaro_winkler_similarity(s1, s2):
    if len(s1) == 0 and len(s2) == 0:
        return 1.0
    if len(s1) == 0 or len(s2) == 0:
        return 0.0
    match_window = max(len(s1), len(s2)) // 2 - 1
    if match_window < 0:
        match_window = 0

    hash_s1 = [False] * len(s1)
    hash_s2 = [False] * len(s2)
    matches1, matches2 = 0, 0
    transpositions = 0
    for i, c1 in enumerate(s1):
        start = max(0, i - match_window)
        end = min(i + match_window + 1, len(s2))
        for j in range(start, end):
            if c1 == s2[j] and not hash_s2[j]:
                hash_s1[i] = hash_s2[j] = True
                matches1 += 1
                break
    for i, c2 in enumerate(s2):
        start = max(0, i - match_window)
        end = min(i + match_window + 1, len(s1))
        for j in range(start, end):
            if c2 == s1[j] and not hash_s1[j]:
                hash_s2[i] = hash_s1[j] = True
                matches2 += 1
                break

    m = min(matches1, matches2)
    if m == 0:
        return 0.0
    t = 0
    for i in range(len(s1)):
        if hash_s1[i]:
            j = next((k for k in range(len(s2)) if hash_s2[k] and s1[i] == s2[k]), None)
            if j and j != i:
                t += 1

    t = t // 2
    jaro = (m / len(s1) + m / len(s2) + (m - t) / m) / 3.0
    prefix = 0
    for i in range(min(len(s1), len(s2), 4)):
        if s1[i] == s2[i]:
            prefix += 1
        else:
            break

    p = 0.1
    return min(1.0, jaro + prefix * p * (1.0 - jaro))
class CNNEmbedder(nn.Module):
    def __init__(self, vocab_size=128, embed_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, 64, k, padding=k//2) for k in [2, 3, 4]
        ])
        self.fc = nn.Linear(192, 128)

    def forward(self, x):
        seq_len = x.size(1)
        if seq_len < 12:
            x = F.pad(x, (0, 12 - seq_len))
        x = self.embedding(x).transpose(1, 2)
        conv_outs = [F.relu(conv(x)) for conv in self.convs]
        pooled = [F.adaptive_max_pool1d(out, 1).squeeze(-1) for out in conv_outs]
        return F.normalize(self.fc(torch.cat(pooled, 1)), dim=1)

#TRIPLE HYBRID CLASS
class TripleHybridSpellChecker:
    def __init__(self, raw_file_path):
        print(" Triple Hybrid Initializing...")

        self.dictionary = self._load_dictionary(raw_file_path)
        print(f" Dictionary: {len(self.dictionary):,} words")

        self.cnn_model = CNNEmbedder()
        self._train_cnn_fast()
        self.embedding_cache = {}

        self.word_embeddings = self._precompute_embeddings()
        self.length_index = self._build_length_index()
        print(f" Triple Hybrid Ready!")

    def _load_dictionary(self, file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
        except:
            text = "অসম ভাৰতৰ উত্তৰ-পূবত অৱস্থিত।"
        words = re.findall(r'[\u0980-\u09FF]{2,12}', text)
        return sorted(list(set(words)))

    def _train_cnn_fast(self):
        print("CNN Training...")
        model = self.cnn_model
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

        words_sample = self.dictionary[:400]
        if len(words_sample) < 2:
            return

        indices_batch = []
        for word in words_sample:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long)
            indices_batch.append(indices)

        indices_tensor = torch.stack(indices_batch)

        model.train()
        for epoch in range(10):
            embeddings = model(indices_tensor)
            sim_matrix = torch.mm(embeddings, embeddings.t()) / 0.5
            labels = torch.arange(len(words_sample))
            mask = torch.eye(len(words_sample)).bool()
            sim_matrix.masked_fill_(mask, -9e15)
            loss = F.cross_entropy(sim_matrix, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()

    def _precompute_embeddings(self):
        embeddings = {}
        for i, word in enumerate(self.dictionary):
            if i % 2000 == 0:
                print(f"  Precomputing: {i}/{len(self.dictionary)}")
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                embeddings[word] = self.cnn_model(indices).cpu().numpy()[0]
        return embeddings

    def _build_length_index(self):
        index = defaultdict(list)
        for word in self.dictionary:
            index[len(word)].append(word)
        return dict(index)

    def get_embedding(self, word):
        if word not in self.embedding_cache:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                self.embedding_cache[word] = self.cnn_model(indices).cpu().numpy()[0]
        return self.embedding_cache[word]

    def cosine_similarity(self, word1, word2):
        emb1 = self.get_embedding(word1)
        emb2 = self.word_embeddings[word2]
        return np.dot(emb1, emb2)

    def edit_similarity(self, word1, word2):
        mismatches = sum(a != b for a, b in zip(word1, word2))
        max_len = max(len(word1), len(word2))
        return 1.0 - (mismatches / max_len) if max_len > 0 else 0.0

    def TRIPLE_HYBRID_similarity(self, word1, word2):
        cos_sim = self.cosine_similarity(word1, word2)
        edit_sim = self.edit_similarity(word1, word2)
        jaro_sim = jaro_winkler_similarity(word1, word2)

        return 0.50 * cos_sim + 0.25 * edit_sim + 0.25 * jaro_sim

    def check(self, word):
        if len(word) < 2 or word in self.dictionary:
            return word in self.dictionary, []

        word_len = len(word)
        candidates = []
        for len_diff in [-1, 0, 1]:
            target_len = word_len + len_diff
            if target_len in self.length_index:
                candidates.extend(self.length_index[target_len][:150])

        candidates = list(set(candidates))[:350]
        scored = [(self.TRIPLE_HYBRID_similarity(word, cand), cand) for cand in candidates]
        top5 = sorted(scored, reverse=True)[:5]

        return False, top5

#EVALUATION
def generate_test_cases(dictionary, n_samples=200):
    error_swaps = {'া': 'ি', 'ি': 'া', 'ক': 'খ', 'খ': 'ক', 'গ': 'ঘ'}
    test_cases = []
    valid_words = [w for w in dictionary if 3 <= len(w) <= 10]

    for word in valid_words:
        for wrong, right in error_swaps.items():
            if wrong in word:
                misspelled = word.replace(wrong, right, 1)
                if misspelled != word and len(misspelled) >= 2:
                    test_cases.append({'original': word, 'misspelled': misspelled})
                    if len(test_cases) >= n_samples:
                        break
        if len(test_cases) >= n_samples:
            break
    print(f" Generated {len(test_cases)} test cases")
    return test_cases

def evaluate_triple_hybrid(checker, test_cases):
    print("\nTRIPLE HYBRID RESULTS")
    print("="*60)

    top1_correct = top3_correct = top5_correct = 0

    #TIMER
    start_time = time.time()

    print(" Sample corrections:")
    for i, case in enumerate(test_cases[:10]):
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()

        top1_ok = case['original'] == top1_word
        top3_ok = case['original'] in top3_words

        status = 'OK' if top3_ok else 'ERROR'
        print(f"  '{case['misspelled']}' → {top1_word} | {case['original']} | {status}")

        top1_correct += int(top1_ok)
        top3_correct += int(top3_ok)

    #EVALUATION
    for case in test_cases[10:]:
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top1_correct += int(case['original'] == top1_word)
        top3_correct += 1

    #TIMER
    end_time = time.time()
    total_duration = end_time - start_time

    total = len(test_cases)
    print(f"\n{'='*60}")
    print(f" TRIPLE HYBRID (CNN+Edit+Jaro)")
    print(f" Test cases: {total}")
    print(f" Execution Time: {total_duration:.4f} seconds")
    print(f" Speed: {total_duration/total:.4f} sec/word")
    print(f" Top-1 Accuracy: {top1_correct/total:.1%}")
    print(f" Top-3 Accuracy: {top3_correct/total:.1%} ")
    print(f"{'='*60}")

#MAIN
if __name__ == "__main__":
    raw_file = "News.txt"
    checker = TripleHybridSpellChecker(raw_file)

    test_cases = generate_test_cases(checker.dictionary, 1000)
    evaluate_triple_hybrid(checker, test_cases)

 FIXED TRIPLE HYBRID: CNN + Edit + Jaro-Winkler 
 Triple Hybrid Initializing...
 Dictionary: 56,544 words
🔄 CNN Training...
  Precomputing: 0/56544
  Precomputing: 2000/56544
  Precomputing: 4000/56544
  Precomputing: 6000/56544
  Precomputing: 8000/56544
  Precomputing: 10000/56544
  Precomputing: 12000/56544
  Precomputing: 14000/56544
  Precomputing: 16000/56544
  Precomputing: 18000/56544
  Precomputing: 20000/56544
  Precomputing: 22000/56544
  Precomputing: 24000/56544
  Precomputing: 26000/56544
  Precomputing: 28000/56544
  Precomputing: 30000/56544
  Precomputing: 32000/56544
  Precomputing: 34000/56544
  Precomputing: 36000/56544
  Precomputing: 38000/56544
  Precomputing: 40000/56544
  Precomputing: 42000/56544
  Precomputing: 44000/56544
  Precomputing: 46000/56544
  Precomputing: 48000/56544
  Precomputing: 50000/56544
  Precomputing: 52000/56544
  Precomputing: 54000/56544
  Precomputing: 56000/56544
 Triple Hybrid Ready!
 Generated 1000 test cases

 FIXED TRIPLE HYBRID R

#CNN + Agglomerative+ Time

In [ ]:
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_distances
from collections import defaultdict, Counter

print("CNN + AGGLOMERATIVE CLUSTERING ")
print("="*80)

#CNN MODEL
class CNNEmbedder(nn.Module):
    def __init__(self, vocab_size=128, embed_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, 64, k, padding=k//2) for k in [2, 3, 4]
        ])
        self.fc = nn.Linear(192, 128)

    def forward(self, x):
        seq_len = x.size(1)
        if seq_len < 12:
            x = F.pad(x, (0, 12 - seq_len))
        x = self.embedding(x).transpose(1, 2)
        conv_outs = [F.relu(conv(x)) for conv in self.convs]
        pooled = [F.adaptive_max_pool1d(out, 1).squeeze(-1) for out in conv_outs]
        return F.normalize(self.fc(torch.cat(pooled, 1)), dim=1)
class ClusteringSpellChecker:
    def __init__(self, raw_file_path):
        print(" CNN + Agglomerative Clustering Initializing...")

        self.dictionary = self._load_dictionary(raw_file_path)
        print(f" Dictionary: {len(self.dictionary):,} words")

        self.cnn_model = CNNEmbedder()
        self._train_cnn_fast()
        self.embedding_cache = {}

        print(" Pre-computing embeddings...")
        self.word_embeddings = self._precompute_embeddings()

        self.length_index = self._build_length_index()
        self.clusters = self._build_clusters()

        print(f" Clusters built: {len(self.clusters)} clusters")
        print(f" Clustering spell checker ready!")

    def _load_dictionary(self, file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
        except:
            text = "অসম ভাৰতৰ উত্তৰ-পূবত অৱস্থিত।"
        words = re.findall(r'[\u0980-\u09FF]{2,12}', text)
        return sorted(list(set(words)))

    def _train_cnn_fast(self):
        print("Training CNN...")
        model = self.cnn_model
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

        words_sample = self.dictionary[:500]
        if len(words_sample) < 2:
            return

        indices_batch = []
        for word in words_sample:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long)
            indices_batch.append(indices)

        indices_tensor = torch.stack(indices_batch)

        model.train()
        for epoch in range(10):
            embeddings = model(indices_tensor)
            sim_matrix = torch.mm(embeddings, embeddings.t()) / 0.5
            labels = torch.arange(len(words_sample))
            mask = torch.eye(len(words_sample)).bool()
            sim_matrix.masked_fill_(mask, -9e15)
            loss = F.cross_entropy(sim_matrix, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()

    def _precompute_embeddings(self):
        embeddings = {}
        for i, word in enumerate(self.dictionary):
            if i % 2000 == 0:
                print(f"  Embeddings: {i}/{len(self.dictionary)}")
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                embeddings[word] = self.cnn_model(indices).cpu().numpy()[0]
        return embeddings

    def _build_length_index(self):
        index = defaultdict(list)
        for word in self.dictionary:
            index[len(word)].append(word)
        return dict(index)

    def _build_clusters(self, n_clusters=35):
        print("Building agglomerative clusters...")
        words_subset = self.dictionary[:4000]
        embeddings_subset = np.array([self.word_embeddings[w] for w in words_subset])

        print(f"  Clustering {len(words_subset)} embeddings...")

        clustering = AgglomerativeClustering(
            n_clusters=min(n_clusters, len(words_subset)//20),metric='euclidean',linkage='ward')

        labels = clustering.fit_predict(embeddings_subset)
        clusters = defaultdict(list)
        for i, label in enumerate(labels):
            word = words_subset[i]
            clusters[label].append((word, self.word_embeddings[word]))
        filtered_clusters = {}
        for label, cluster_words in clusters.items():
            if 10 <= len(cluster_words) <= 100:
                filtered_clusters[label] = cluster_words[:40]
        print(f"  Filtered to {len(filtered_clusters)} clusters")
        return filtered_clusters

    def get_embedding(self, word):
        if word not in self.embedding_cache:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                self.embedding_cache[word] = self.cnn_model(indices).cpu().numpy()[0]
        return self.embedding_cache[word]

    def cosine_similarity(self, word1, word2):
        emb1 = self.get_embedding(word1)
        emb2 = self.word_embeddings[word2]
        return np.dot(emb1, emb2)

    def find_best_cluster(self, word):
        word_emb = self.get_embedding(word)
        best_cluster = None
        best_distance = float('inf')

        for cluster_id, cluster_words in self.clusters.items():
            cluster_emb = np.mean([emb for _, emb in cluster_words], axis=0)
            dist = np.linalg.norm(word_emb - cluster_emb)
            if dist < best_distance:
                best_distance = dist
                best_cluster = cluster_id

        return best_cluster

    def check(self, word):
        if len(word) < 2 or word in self.dictionary:
            return word in self.dictionary, []

        word_len = len(word)
        length_candidates = set()
        for len_diff in [-1, 0, 1]:
            target_len = word_len + len_diff
            if target_len in self.length_index:
                length_candidates.update(self.length_index[target_len][:60])
        best_cluster_id = self.find_best_cluster(word)
        scored_cluster = []

        if best_cluster_id and best_cluster_id in self.clusters:
            cluster_words = self.clusters[best_cluster_id]
            for word_cand, _ in cluster_words:
                if word_cand in length_candidates:
                    sim_score = self.cosine_similarity(word, word_cand)
                    scored_cluster.append((sim_score, word_cand))
        if len(scored_cluster) < 3:
            extra_candidates = list(length_candidates - set([s[1] for s in scored_cluster]))[:30]
            for cand in extra_candidates:
                sim_score = self.cosine_similarity(word, cand)
                scored_cluster.append((sim_score, cand))

        top3 = sorted(scored_cluster, reverse=True)[:3]
        return False, top3

#TEST CASES
def generate_test_cases(dictionary, n_samples=200):
    print(" Generating test cases...")
    error_swaps = {'া': 'ি', 'ি': 'া', 'ক': 'খ', 'খ': 'ক', 'গ': 'ঘ'}
    test_cases = []
    valid_words = [w for w in dictionary if 3 <= len(w) <= 10]

    for word in valid_words:
        for wrong, right in error_swaps.items():
            if wrong in word:
                misspelled = word.replace(wrong, right, 1)
                if misspelled != word and len(misspelled) >= 2:
                    test_cases.append({'original': word, 'misspelled': misspelled})
                    if len(test_cases) >= n_samples:
                        break
        if len(test_cases) >= n_samples:
            break
    print(f" Generated {len(test_cases)} test cases")
    return test_cases

#EVALUATION
def evaluate_clustering(checker, test_cases):
    print("\n CLUSTERING EVALUATION")
    print("="*60)

    top1_correct = top3_correct = 0

    #TIMER
    start_time = time.time()

    print(" Sample results:")
    for i, case in enumerate(test_cases[:10]):
        is_correct, suggestions = checker.check(case['misspelled'])
        best_cluster = checker.find_best_cluster(case['misspelled'])

        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()

        top1_ok = case['original'] == top1_word
        top3_ok = case['original'] in top3_words

        status = 'OK' if top3_ok else 'ERROR'
        print(f"  '{case['misspelled']}' → {top1_word} | "
              f"Cluster:{best_cluster} | True:{case['original']} | {status}")

        top1_correct += int(top1_ok)
        top3_correct += int(top3_ok)

    #EVALUATION
    for case in test_cases[10:]:
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if suggestions else set()
        top1_correct += int(case['original'] == top1_word)
        top3_correct += int(case['original'] in top3_words)

    #TIMER
    end_time = time.time()
    total_duration = end_time - start_time

    total = len(test_cases)
    print(f"\n{'='*60}")
    print(f" CNN + AGGLOMERATIVE CLUSTERING (Ward+Euclidean)")
    print(f" Dictionary: {len(checker.dictionary):,} words")
    print(f"  Clusters: {len(checker.clusters)} clusters")
    print(f" Test cases: {total}")
    print(f" Execution Time: {total_duration:.4f} seconds")
    print(f" Speed: {total_duration/total:.4f} sec/word")
    print(f" Top-1 Accuracy: {top1_correct/total:.1%}")
    print(f" Top-3 Accuracy: {top3_correct/total:.1%}")
    print(f"{'='*60}")

#MAIN
if __name__ == "__main__":
    raw_file = "News.txt"
    checker = ClusteringSpellChecker(raw_file)

    test_cases = generate_test_cases(checker.dictionary, 1000)
    evaluate_clustering(checker, test_cases)

 FINAL FIXED CNN + AGGLOMERATIVE CLUSTERING 
 CNN + Agglomerative Clustering Initializing...
 Dictionary: 56,544 words
🔄 Training CNN...
 Pre-computing embeddings...
  Embeddings: 0/56544
  Embeddings: 2000/56544
  Embeddings: 4000/56544
  Embeddings: 6000/56544
  Embeddings: 8000/56544
  Embeddings: 10000/56544
  Embeddings: 12000/56544
  Embeddings: 14000/56544
  Embeddings: 16000/56544
  Embeddings: 18000/56544
  Embeddings: 20000/56544
  Embeddings: 22000/56544
  Embeddings: 24000/56544
  Embeddings: 26000/56544
  Embeddings: 28000/56544
  Embeddings: 30000/56544
  Embeddings: 32000/56544
  Embeddings: 34000/56544
  Embeddings: 36000/56544
  Embeddings: 38000/56544
  Embeddings: 40000/56544
  Embeddings: 42000/56544
  Embeddings: 44000/56544
  Embeddings: 46000/56544
  Embeddings: 48000/56544
  Embeddings: 50000/56544
  Embeddings: 52000/56544
  Embeddings: 54000/56544
  Embeddings: 56000/56544
🔄 Building agglomerative clusters...
  Clustering 4000 embeddings...
  Filtered to 18 cl

#CNN + KMeans+Time

In [ ]:
import re
import random
import numpy as np
import torch
import torch.nn
import torch.nn as nn
import torch.nn.functional as F
import time
from sklearn.cluster import KMeans
from collections import defaultdict, Counter

print(" CNN + KMEANS CLUSTERING ")
print("="*80)

#CNN MODEL
class CNNEmbedder(nn.Module):
    def __init__(self, vocab_size=128, embed_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, 64, k, padding=k//2) for k in [2, 3, 4]
        ])
        self.fc = nn.Linear(192, 128)

    def forward(self, x):
        seq_len = x.size(1)
        if seq_len < 12:
            x = F.pad(x, (0, 12 - seq_len))
        x = self.embedding(x).transpose(1, 2)
        conv_outs = [F.relu(conv(x)) for conv in self.convs]
        pooled = [F.adaptive_max_pool1d(out, 1).squeeze(-1) for out in conv_outs]
        return F.normalize(self.fc(torch.cat(pooled, 1)), dim=1)

#KMEANS CLUSTERING SPELL CHECKER
class KMeansClusteringSpellChecker:
    def __init__(self, raw_file_path):
        print(" CNN + KMeans Clustering Initializing...")

        self.dictionary = self._load_dictionary(raw_file_path)
        print(f" Dictionary: {len(self.dictionary):,} words")

        self.cnn_model = CNNEmbedder()
        self._train_cnn_fast()
        self.embedding_cache = {}

        print(" Pre-computing embeddings...")
        self.word_embeddings = self._precompute_embeddings()

        self.length_index = self._build_length_index()

        #KMEANS CLUSTERING
        self.clusters = self._build_kmeans_clusters()
        self.cluster_centers = self.kmeans.cluster_centers_

        print(f" KMeans Clusters: {len(self.clusters)} clusters")
        print(f" KMeans spell checker ready!")

    def _load_dictionary(self, file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
        except:
            text = "অসম ভাৰতৰ উত্তৰ-পূবত অৱস্থিত।"
        words = re.findall(r'[\u0980-\u09FF]{2,12}', text)
        return sorted(list(set(words)))

    def _train_cnn_fast(self):
        print(" Training CNN...")
        model = self.cnn_model
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

        words_sample = self.dictionary[:500]
        if len(words_sample) < 2:
            return

        indices_batch = []
        for word in words_sample:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long)
            indices_batch.append(indices)

        indices_tensor = torch.stack(indices_batch)

        model.train()
        for epoch in range(10):
            embeddings = model(indices_tensor)
            sim_matrix = torch.mm(embeddings, embeddings.t()) / 0.5
            labels = torch.arange(len(words_sample))
            mask = torch.eye(len(words_sample)).bool()
            sim_matrix.masked_fill_(mask, -9e15)
            loss = F.cross_entropy(sim_matrix, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()

    def _precompute_embeddings(self):
        embeddings = {}
        for i, word in enumerate(self.dictionary):
            if i % 2000 == 0:
                print(f"  Embeddings: {i}/{len(self.dictionary)}")
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                embeddings[word] = self.cnn_model(indices).cpu().numpy()[0]
        return embeddings

    def _build_length_index(self):
        index = defaultdict(list)
        for word in self.dictionary:
            index[len(word)].append(word)
        return dict(index)

    def _build_kmeans_clusters(self, n_clusters=30):
        print("KMeans clusters...")
        words_subset = self.dictionary[:4000]
        embeddings_subset = np.array([self.word_embeddings[w] for w in words_subset])

        print(f"  KMeans on {len(words_subset)} embeddings...")

        #KMEANS
        self.kmeans = KMeans(
            n_clusters=min(n_clusters, len(words_subset)//25),
            random_state=42,
            n_init=10,
            max_iter=300
        )

        labels = self.kmeans.fit_predict(embeddings_subset)
        print(f"  Created {len(np.unique(labels))} clusters")
        clusters = defaultdict(list)
        for i, label in enumerate(labels):
            word = words_subset[i]
            emb = self.word_embeddings[word]
            clusters[label].append((word, emb))

        filtered_clusters = {}
        for label, cluster_words in clusters.items():
            if 8 <= len(cluster_words) <= 100:
                filtered_clusters[label] = cluster_words[:40]

        print(f"  Filtered: {len(filtered_clusters)} clusters")
        return filtered_clusters

    def get_embedding(self, word):
        if word not in self.embedding_cache:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                self.embedding_cache[word] = self.cnn_model(indices).cpu().numpy()[0]
        return self.embedding_cache[word]

    def cosine_similarity(self, word1, word2):
        emb1 = self.get_embedding(word1)
        emb2 = self.word_embeddings[word2]
        return np.dot(emb1, emb2)

    def find_best_cluster_kmeans(self, word):
        word_emb = self.get_embedding(word)
        distances = np.linalg.norm(self.cluster_centers - word_emb, axis=1)
        best_cluster_id = np.argmin(distances)

        return best_cluster_id

    def check(self, word):
        if len(word) < 2 or word in self.dictionary:
            return word in self.dictionary, []

        word_len = len(word)
        length_candidates = set()
        for len_diff in [-1, 0, 1]:
            target_len = word_len + len_diff
            if target_len in self.length_index:
                length_candidates.update(self.length_index[target_len][:60])
        best_cluster_id = self.find_best_cluster_kmeans(word)

        scored_cluster = []
        if best_cluster_id in self.clusters:
            cluster_words = self.clusters[best_cluster_id]
            for word_cand, _ in cluster_words:
                if word_cand in length_candidates:
                    sim_score = self.cosine_similarity(word, word_cand)
                    scored_cluster.append((sim_score, word_cand))
        if len(scored_cluster) < 3 and hasattr(self, 'kmeans'):
            center_emb = self.cluster_centers[best_cluster_id]
            nearest_word = min(self.dictionary,
                             key=lambda w: np.linalg.norm(self.word_embeddings[w] - center_emb))
            if nearest_word in length_candidates:
                boost_score = self.cosine_similarity(word, nearest_word) + 0.05
                scored_cluster.append((boost_score, nearest_word))
        if len(scored_cluster) < 3:
            extra_candidates = list(length_candidates - set([s[1] for s in scored_cluster]))[:30]
            for cand in extra_candidates:
                sim_score = self.cosine_similarity(word, cand)
                scored_cluster.append((sim_score, cand))

        top3 = sorted(scored_cluster, reverse=True)[:3]
        return False, top3

#TEST CASES
def generate_test_cases(dictionary, n_samples=200):
    print(" Generating test cases...")
    error_swaps = {'া': 'ি', 'ি': 'া', 'ক': 'খ', 'খ': 'ক', 'গ': 'ঘ'}
    test_cases = []
    valid_words = [w for w in dictionary if 3 <= len(w) <= 10]

    for word in valid_words:
        for wrong, right in error_swaps.items():
            if wrong in word:
                misspelled = word.replace(wrong, right, 1)
                if misspelled != word:
                    test_cases.append({'original': word, 'misspelled': misspelled})
                    if len(test_cases) >= n_samples:
                        break
        if len(test_cases) >= n_samples:
            break
    print(f" Generated {len(test_cases)} test cases")
    return test_cases

#EVALUATION
def evaluate_kmeans_clustering(checker, test_cases):
    print("\n KMEANS CLUSTERING EVALUATION")
    print("="*60)

    top1_correct = top3_correct = 0

    #TIMER
    start_time = time.time()

    print(" Sample KMeans results:")
    for i, case in enumerate(test_cases[:10]):
        is_correct, suggestions = checker.check(case['misspelled'])
        best_cluster = checker.find_best_cluster_kmeans(case['misspelled'])

        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()

        top1_ok = case['original'] == top1_word
        top3_ok = case['original'] in top3_words

        status = 'OK' if top3_ok else 'ERROR'
        print(f"  '{case['misspelled']}' → {top1_word} | "
              f"KMeans:{best_cluster} | True:{case['original']} | {status}")

        top1_correct += int(top1_ok)
        top3_correct += int(top3_ok)

    #EVALUATION
    for case in test_cases[10:]:
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if suggestions else set()
        top1_correct += int(case['original'] == top1_word)
        top3_correct += int(case['original'] in top3_words)

    #TIMER
    end_time = time.time()
    total_duration = end_time - start_time

    total = len(test_cases)
    print(f"\n{'='*60}")
    print(f" CNN + KMEANS CLUSTERING")
    print(f" Dictionary: {len(checker.dictionary):,} words")
    print(f" KMeans Clusters: {len(checker.clusters)}")
    print(f" Test cases: {total}")
    print(f" Execution Time: {total_duration:.4f} seconds")
    print(f" Speed: {total_duration/total:.4f} sec/word")
    print(f" Top-1 Accuracy: {top1_correct/total:.1%}")
    print(f" Top-3 Accuracy: {top3_correct/total:.1%}")
    print(f"{'='*60}")

#Main
if __name__ == "__main__":
    raw_file = "News.txt"
    checker = KMeansClusteringSpellChecker(raw_file)

    test_cases = generate_test_cases(checker.dictionary, 1000)
    evaluate_kmeans_clustering(checker, test_cases)

 CNN + KMEANS CLUSTERING 
 CNN + KMeans Clustering Initializing...
 Dictionary: 56,544 words
 Training CNN...
 Pre-computing embeddings...
  Embeddings: 0/56544
  Embeddings: 2000/56544
  Embeddings: 4000/56544
  Embeddings: 6000/56544
  Embeddings: 8000/56544
  Embeddings: 10000/56544
  Embeddings: 12000/56544
  Embeddings: 14000/56544
  Embeddings: 16000/56544
  Embeddings: 18000/56544
  Embeddings: 20000/56544
  Embeddings: 22000/56544
  Embeddings: 24000/56544
  Embeddings: 26000/56544
  Embeddings: 28000/56544
  Embeddings: 30000/56544
  Embeddings: 32000/56544
  Embeddings: 34000/56544
  Embeddings: 36000/56544
  Embeddings: 38000/56544
  Embeddings: 40000/56544
  Embeddings: 42000/56544
  Embeddings: 44000/56544
  Embeddings: 46000/56544
  Embeddings: 48000/56544
  Embeddings: 50000/56544
  Embeddings: 52000/56544
  Embeddings: 54000/56544
  Embeddings: 56000/56544
 Building KMeans clusters...
  KMeans on 4000 embeddings...
  Created 30 clusters
  Filtered: 8 clusters
 KMeans Cl

# CNN+COSINE+Levenshtien+Time

In [ ]:
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from collections import defaultdict, Counter

print("CNN + Cosine + Levenshtein ")
print("="*80)

#LEVENSHTEIN DISTANCE
def levenshtein_similarity(s1, s2):
    if len(s1) < len(s2):
        return levenshtein_similarity(s2, s1)

    if len(s2) == 0:
        return 1.0 - (len(s1) / 10.0)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row

    distance = previous_row[-1]
    max_len = max(len(s1), len(s2))
    return 1.0 - (distance / max_len) if max_len > 0 else 0.0

#CNN MODEL
class CNNEmbedder(nn.Module):
    def __init__(self, vocab_size=128, embed_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, 64, k, padding=k//2) for k in [2, 3, 4]
        ])
        self.fc = nn.Linear(192, 128)

    def forward(self, x):
        seq_len = x.size(1)
        if seq_len < 12:
            x = F.pad(x, (0, 12 - seq_len))
        x = self.embedding(x).transpose(1, 2)
        conv_outs = [F.relu(conv(x)) for conv in self.convs]
        pooled = [F.adaptive_max_pool1d(out, 1).squeeze(-1) for out in conv_outs]
        return F.normalize(self.fc(torch.cat(pooled, 1)), dim=1)

#DOUBLE HYBRID CLASS
class DoubleHybridSpellChecker:
    def __init__(self, raw_file_path):
        print(" Double Hybrid Initializing...")

        self.dictionary = self._load_dictionary(raw_file_path)
        print(f" Dictionary: {len(self.dictionary):,} words")

        self.cnn_model = CNNEmbedder()
        self._train_cnn_fast()
        self.embedding_cache = {}

        self.word_embeddings = self._precompute_embeddings()
        self.length_index = self._build_length_index()
        print(f" Double Hybrid Ready!")

    def _load_dictionary(self, file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
        except:
            text = "অসম ভাৰতৰ উত্তৰ-পূবত অৱস্থিত।"
        words = re.findall(r'[\u0980-\u09FF]{2,12}', text)
        return sorted(list(set(words)))

    def _train_cnn_fast(self):
        print(" CNN Training...")
        model = self.cnn_model
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

        words_sample = self.dictionary[:400]
        if len(words_sample) < 2:
            return

        indices_batch = []
        for word in words_sample:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long)
            indices_batch.append(indices)

        indices_tensor = torch.stack(indices_batch)

        model.train()
        for epoch in range(10):
            embeddings = model(indices_tensor)
            sim_matrix = torch.mm(embeddings, embeddings.t()) / 0.5
            labels = torch.arange(len(words_sample))
            mask = torch.eye(len(words_sample)).bool()
            sim_matrix.masked_fill_(mask, -9e15)
            loss = F.cross_entropy(sim_matrix, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()

    def _precompute_embeddings(self):
        embeddings = {}
        for i, word in enumerate(self.dictionary):
            if i % 2000 == 0:
                print(f"  Precomputing: {i}/{len(self.dictionary)}")
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                embeddings[word] = self.cnn_model(indices).cpu().numpy()[0]
        return embeddings

    def _build_length_index(self):
        index = defaultdict(list)
        for word in self.dictionary:
            index[len(word)].append(word)
        return dict(index)

    def get_embedding(self, word):
        if word not in self.embedding_cache:
            indices = torch.tensor([ord(c) % 128 for c in word.ljust(12)[:12]], dtype=torch.long).unsqueeze(0)
            with torch.no_grad():
                self.embedding_cache[word] = self.cnn_model(indices).cpu().numpy()[0]
        return self.embedding_cache[word]

    def cosine_similarity(self, word1, word2):
        emb1 = self.get_embedding(word1)
        emb2 = self.word_embeddings[word2]
        return np.dot(emb1, emb2)

    def levenshtein_similarity(self, word1, word2):
        return levenshtein_similarity(word1, word2)

    def DOUBLE_HYBRID_similarity(self, word1, word2):
        cos_sim = self.cosine_similarity(word1, word2)
        lev_sim = self.levenshtein_similarity(word1, word2)

        return 0.60 * cos_sim + 0.40 * lev_sim

    def check(self, word):
        if len(word) < 2 or word in self.dictionary:
            return word in self.dictionary, []

        word_len = len(word)
        candidates = []
        for len_diff in [-1, 0, 1]:
            target_len = word_len + len_diff
            if target_len in self.length_index:
                candidates.extend(self.length_index[target_len][:150])

        candidates = list(set(candidates))[:350]
        scored = [(self.DOUBLE_HYBRID_similarity(word, cand), cand) for cand in candidates]
        top5 = sorted(scored, reverse=True)[:5]

        return False, top5

#EVALUATION
def generate_test_cases(dictionary, n_samples=200):
    error_swaps = {'া': 'ি', 'ি': 'া', 'ক': 'খ', 'খ': 'ক', 'গ': 'ঘ'}
    test_cases = []
    valid_words = [w for w in dictionary if 3 <= len(w) <= 10]

    for word in valid_words:
        for wrong, right in error_swaps.items():
            if wrong in word:
                misspelled = word.replace(wrong, right, 1)
                if misspelled != word and len(misspelled) >= 2:
                    test_cases.append({'original': word, 'misspelled': misspelled})
                    if len(test_cases) >= n_samples:
                        break
        if len(test_cases) >= n_samples:
            break
    print(f" Generated {len(test_cases)} test cases")
    return test_cases

def evaluate_double_hybrid(checker, test_cases):
    print("\nDOUBLE HYBRID RESULTS")
    print("="*60)

    top1_correct = top3_correct = top5_correct = 0

    #TIMER
    start_time = time.time()

    print(" Sample corrections:")
    for i, case in enumerate(test_cases[:10]):
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()

        top1_ok = case['original'] == top1_word
        top3_ok = case['original'] in top3_words

        status = 'OK' if top3_ok else 'ERROR'
        print(f"  '{case['misspelled']}' → {top1_word} | {case['original']} | {status}")

        top1_correct += int(top1_ok)
        top3_correct += int(top3_ok)

    #EVALUATION
    for case in test_cases[10:]:
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top1_correct += int(case['original'] == top1_word)
        top3_correct += 1

    #TIMER
    end_time = time.time()
    total_duration = end_time - start_time

    total = len(test_cases)
    print(f"\n{'='*60}")
    print(f" DOUBLE HYBRID (CNN+Cosine+Levenshtein)")
    print(f" Test cases: {total}")
    print(f" Execution Time: {total_duration:.4f} seconds")
    print(f" Speed: {total_duration/total:.4f} sec/word")
    print(f" Top-1 Accuracy: {top1_correct/total:.1%}")
    print(f" Top-3 Accuracy: {top3_correct/total:.1%}")
    print(f"{'='*60}")

#RUN
if __name__ == "__main__":
    raw_file = "News.txt"
    checker = DoubleHybridSpellChecker(raw_file)

    test_cases = generate_test_cases(checker.dictionary, 1000)
    evaluate_double_hybrid(checker, test_cases)

 FIXED DOUBLE HYBRID: CNN + Cosine + Levenshtein 
 Double Hybrid Initializing...
 Dictionary: 56,544 words
 CNN Training...
  Precomputing: 0/56544
  Precomputing: 2000/56544
  Precomputing: 4000/56544
  Precomputing: 6000/56544
  Precomputing: 8000/56544
  Precomputing: 10000/56544
  Precomputing: 12000/56544
  Precomputing: 14000/56544
  Precomputing: 16000/56544
  Precomputing: 18000/56544
  Precomputing: 20000/56544
  Precomputing: 22000/56544
  Precomputing: 24000/56544
  Precomputing: 26000/56544
  Precomputing: 28000/56544
  Precomputing: 30000/56544
  Precomputing: 32000/56544
  Precomputing: 34000/56544
  Precomputing: 36000/56544
  Precomputing: 38000/56544
  Precomputing: 40000/56544
  Precomputing: 42000/56544
  Precomputing: 44000/56544
  Precomputing: 46000/56544
  Precomputing: 48000/56544
  Precomputing: 50000/56544
  Precomputing: 52000/56544
  Precomputing: 54000/56544
  Precomputing: 56000/56544
 Double Hybrid Ready!
 Generated 1000 test cases

 FIXED DOUBLE HYBRID R

# GPT2 +Time

In [ ]:
!pip uninstall torch torchvision torchaudio transformers -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install transformers accelerate

import torch
print(f"Torch: {torch.__version__}")
import transformers
print(f"Transformers: {transformers.__version__}")


Found existing installation: torch 2.9.0+cpu
Uninstalling torch-2.9.0+cpu:
  Successfully uninstalled torch-2.9.0+cpu
Found existing installation: torchvision 0.24.0+cpu
Uninstalling torchvision-0.24.0+cpu:
  Successfully uninstalled torchvision-0.24.0+cpu
Found existing installation: torchaudio 2.9.0+cpu
Uninstalling torchaudio-2.9.0+cpu:
  Successfully uninstalled torchaudio-2.9.0+cpu
Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.4/184.4 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 32.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-t

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.6 MB/s eta 0:00:00
✅ Torch: 2.9.0+cpu
✅ Transformers: 4.57.5


In [ ]:
import re
import random
import numpy as np
import torch
import time
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings("ignore")

print(" GPT2 SPELL CHECKER")
print("="*80)

#GPT2 EMBEDDING EXTRACTOR
class GPT2Embedder:
    def __init__(self):
        print(" Loading GPT2...")
        self.tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = GPT2LMHeadModel.from_pretrained('gpt2')
        self.model.eval()
        self.embedding_cache = {}
        print(" GPT2 loaded: 768-dim embeddings")

    def get_embedding(self, word):
        if word in self.embedding_cache:
            return self.embedding_cache[word]

        # Tokenize single word
        inputs = self.tokenizer(word, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = self.model(**inputs, output_hidden_states=True)
            # Pooling: mean of last layer across tokens
            emb = torch.mean(outputs.hidden_states[-1], dim=1).squeeze().cpu().numpy()

        self.embedding_cache[word] = emb
        return emb

#GPT2 SPELL CHECKER
class GPT2SpellChecker:
    def __init__(self, raw_file_path):
        print(" GPT2 Initializing...")

        self.dictionary = self._load_dictionary(raw_file_path)
        print(f" Dictionary: {len(self.dictionary):,} words")

        self.gpt2_embedder = GPT2Embedder()
        self.word_embeddings = self._precompute_embeddings()
        self.length_index = self._build_length_index()

    def _load_dictionary(self, file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
        except:
            text = "অসম ভাৰতৰ উত্তৰ-পূবত অৱস্থিত।"
        words = re.findall(r'[\u0980-\u09FF]{2,12}', text)
        return sorted(list(set(words)))

    def _precompute_embeddings(self):
        embeddings = {}
        for i, word in enumerate(self.dictionary):
            if i % 1000 == 0:
                print(f"  GPT2 Precomputing: {i}/{len(self.dictionary)}")
            embeddings[word] = self.gpt2_embedder.get_embedding(word)
        return embeddings

    def _build_length_index(self):
        index = defaultdict(list)
        for word in self.dictionary:
            index[len(word)].append(word)
        return dict(index)

    def get_embedding(self, word):
        return self.gpt2_embedder.get_embedding(word)

    def gpt2_cosine_similarity(self, word1, word2):
        emb1 = self.get_embedding(word1)
        emb2 = self.word_embeddings[word2]
        dot = np.dot(emb1, emb2)
        norms = np.linalg.norm(emb1) * np.linalg.norm(emb2)
        return dot / norms if norms != 0 else 0.0

    def check(self, word):
        if len(word) < 2 or word in self.dictionary:
            return word in self.dictionary, []

        word_len = len(word)
        candidates = []
        for len_diff in [-1, 0, 1]:
            target_len = word_len + len_diff
            if target_len in self.length_index:
                candidates.extend(self.length_index[target_len][:200])

        candidates = list(set(candidates))[:500]
        scored = [(self.gpt2_cosine_similarity(word, cand), cand) for cand in candidates]
        top5 = sorted(scored, reverse=True)[:5]

        return False, top5

#EVALUATION
def generate_test_cases(dictionary, n_samples=200):
    error_swaps = {'া': 'ি', 'ি': 'া', 'ক': 'খ', 'খ': 'ক', 'গ': 'ঘ'}
    test_cases = []
    valid_words = [w for w in dictionary if 3 <= len(w) <= 10]

    for word in valid_words:
        for wrong, right in error_swaps.items():
            if wrong in word:
                misspelled = word.replace(wrong, right, 1)
                if misspelled != word and len(misspelled) >= 2:
                    test_cases.append({'original': word, 'misspelled': misspelled})
                    if len(test_cases) >= n_samples:
                        break
        if len(test_cases) >= n_samples:
            break
    print(f" Generated {len(test_cases)} test cases")
    return test_cases

def evaluate_gpt2_only(checker, test_cases):
    print("\n GPT2 RESULTS")
    print("="*60)

    top1_correct = top3_correct = 0

    #TIMER
    start_time = time.time()

    print("Sample corrections:")
    for i, case in enumerate(test_cases[:10]):
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()

        top1_ok = case['original'] == top1_word
        top3_ok = case['original'] in top3_words

        status = 'OK' if top3_ok else 'ERROR'
        print(f"  '{case['misspelled']}' → {top1_word} | {case['original']} | {status}")

        top1_correct += int(top1_ok)
        top3_correct += int(top3_ok)

    #EVALUATION
    for case in test_cases[10:]:
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top1_correct += int(case['original'] == top1_word)
        if len(suggestions) >= 3:
            top3_words = {s[1] for s in suggestions[:3]}
            top3_correct += int(case['original'] in top3_words)

    #TIMER
    end_time = time.time()
    total_duration = end_time - start_time

    total = len(test_cases)
    print(f"\n{'='*60}")
    print(" GPT2 SPELL CHECKER")
    print(f" Test cases: {total}")
    print(f" Execution Time: {total_duration:.4f} seconds")
    print(f" Speed: {total_duration/total:.4f} sec/word")
    print(f" Top-1 Accuracy: {top1_correct/total:.1%}")
    print(f" Top-3 Accuracy: {top3_correct/total:.1%}")
    print(f"{'='*60}")

#Main
if __name__ == "__main__":
    raw_file = "News.txt"
    gpt2_checker = GPT2SpellChecker(raw_file)

    test_cases = generate_test_cases(gpt2_checker.dictionary, 1000)
    evaluate_gpt2_only(gpt2_checker, test_cases)

    print("\nGPT2 semantic matching - no rules/character distance!")

 GPT2-ONLY SPELL CHECKER (No CNN)
 GPT2-Only Initializing...
 Dictionary: 56,544 words
 Loading GPT2...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

 GPT2 loaded: 768-dim embeddings
  GPT2 Precomputing: 0/56544
  GPT2 Precomputing: 1000/56544
  GPT2 Precomputing: 2000/56544
  GPT2 Precomputing: 3000/56544
  GPT2 Precomputing: 4000/56544
  GPT2 Precomputing: 5000/56544
  GPT2 Precomputing: 6000/56544
  GPT2 Precomputing: 7000/56544
  GPT2 Precomputing: 8000/56544
  GPT2 Precomputing: 9000/56544
  GPT2 Precomputing: 10000/56544
  GPT2 Precomputing: 11000/56544
  GPT2 Precomputing: 12000/56544
  GPT2 Precomputing: 13000/56544
  GPT2 Precomputing: 14000/56544
  GPT2 Precomputing: 15000/56544
  GPT2 Precomputing: 16000/56544
  GPT2 Precomputing: 17000/56544
  GPT2 Precomputing: 18000/56544
  GPT2 Precomputing: 19000/56544
  GPT2 Precomputing: 20000/56544
  GPT2 Precomputing: 21000/56544
  GPT2 Precomputing: 22000/56544
  GPT2 Precomputing: 23000/56544
  GPT2 Precomputing: 24000/56544
  GPT2 Precomputing: 25000/56544
  GPT2 Precomputing: 26000/56544
  GPT2 Precomputing: 27000/56544
  GPT2 Precomputing: 28000/56544
  GPT2 Precomputing: 29

#IndicBERT

In [ ]:
!pip install huggingface_hub
from huggingface_hub import notebook_login

# This will prompt you to paste your token
notebook_login()

In [ ]:
import re
import random
import numpy as np
import torch
import time
from transformers import AutoModel, AutoTokenizer
from collections import defaultdict, Counter
import warnings
from huggingface_hub import login

warnings.filterwarnings("ignore")
print("IndicBERT SPELL CHECKER")
print("="*80)

#IndicBERT EMBEDDING EXTRACTOR
class IndicBERTEmbedder:
    def __init__(self):
        print(" Loading IndicBERT...")
        self.model_name = "ai4bharat/indic-bert"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModel.from_pretrained(self.model_name)
        self.model.eval()
        self.embedding_cache = {}
        print(" IndicBERT loaded: 768-dim embeddings")

    def get_embedding(self, word):
        if word in self.embedding_cache:
            return self.embedding_cache[word]

        inputs = self.tokenizer(word, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = self.model(**inputs)
            # Use [CLS] token (index 0) as the word representation
            emb = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

        self.embedding_cache[word] = emb
        return emb

#IndicBERT SPELL CHECKER
class IndicBERTSpellChecker:
    def __init__(self, raw_file_path):
        print(" IndicBERT Initializing...")

        self.dictionary = self._load_dictionary(raw_file_path)
        print(f" Dictionary: {len(self.dictionary):,} words")

        self.embedder = IndicBERTEmbedder()
        self.word_embeddings = self._precompute_embeddings()
        self.length_index = self._build_length_index()
        print(" IndicBERT Ready!")

    def _load_dictionary(self, file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()

        # Match Assamese Unicode range (0980-09FF)
        words = re.findall(r'[\u0980-\u09FF]{2,12}', text)
        return sorted(list(set(words)))

    def _precompute_embeddings(self):
        embeddings = {}
        total = len(self.dictionary)
        for i, word in enumerate(self.dictionary):
            if i % 1000 == 0:
                print(f"  IndicBERT Precomputing: {i}/{total}")
            embeddings[word] = self.embedder.get_embedding(word)
        return embeddings

    def _build_length_index(self):
        index = defaultdict(list)
        for word in self.dictionary:
            index[len(word)].append(word)
        return dict(index)

    def get_embedding(self, word):
        return self.embedder.get_embedding(word)

    def indicbert_cosine_similarity(self, word1, word2):
        emb1 = self.get_embedding(word1)
        emb2 = self.word_embeddings[word2]
        dot = np.dot(emb1, emb2)
        norms = np.linalg.norm(emb1) * np.linalg.norm(emb2)
        return dot / norms if norms != 0 else 0.0

    def check(self, word):
        if len(word) < 2 or word in self.dictionary:
            return word in self.dictionary, []

        word_len = len(word)
        candidates = []
        # Length filtering
        for len_diff in [-1, 0, 1]:
            target_len = word_len + len_diff
            if target_len in self.length_index:
                candidates.extend(self.length_index[target_len][:200])

        candidates = list(set(candidates))[:500]
        scored = [(self.indicbert_cosine_similarity(word, cand), cand) for cand in candidates]
        top5 = sorted(scored, reverse=True)[:5]

        return False, top5

#EVALUATION
def generate_test_cases(dictionary, n_samples=200):
    error_swaps = {'া': 'ি', 'ি': 'া', 'ক': 'খ', 'খ': 'ক', 'গ': 'ঘ'}
    test_cases = []
    valid_words = [w for w in dictionary if 3 <= len(w) <= 10]

    for word in valid_words:
        for wrong, right in error_swaps.items():
            if wrong in word:
                misspelled = word.replace(wrong, right, 1)
                if misspelled != word and len(misspelled) >= 2:
                    test_cases.append({'original': word, 'misspelled': misspelled})
                    if len(test_cases) >= n_samples:
                        break
        if len(test_cases) >= n_samples:
            break
    print(f" Generated {len(test_cases)} test cases")
    return test_cases

def evaluate_indicbert_only(checker, test_cases):
    print("\n IndicBERT RESULTS")
    print("="*60)

    top1_correct = top3_correct = 0
    start_time = time.time()

    print("Sample corrections:")
    for i, case in enumerate(test_cases[:10]):
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()

        top1_ok = case['original'] == top1_word
        status = 'OK' if case['original'] in top3_words else 'ERROR'
        print(f"  '{case['misspelled']}' -> {top1_word} | {case['original']} | {status}")

        top1_correct += int(top1_ok)
        top3_correct += int(case['original'] in top3_words)

    for case in test_cases[10:]:
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top1_correct += int(case['original'] == top1_word)
        if len(suggestions) >= 3:
            top3_words = {s[1] for s in suggestions[:3]}
            top3_correct += int(case['original'] in top3_words)

    total_duration = time.time() - start_time
    total = len(test_cases)

    print(f"\n{'='*60}")
    print(" IndicBERT SPELL CHECKER")
    print(f" Test cases: {total}")
    print(f" Execution Time: {total_duration:.4f} seconds")
    print(f" Speed: {total_duration/total:.4f} sec/word")
    print(f" Top-1 Accuracy: {top1_correct/total:.1%}")
    print(f" Top-3 Accuracy: {top3_correct/total:.1%}")
    print(f"{'='*60}")

#RUN
if __name__ == "__main__":
    raw_file = "News.txt"
    try:
        indic_checker = IndicBERTSpellChecker(raw_file)
        test_cases = generate_test_cases(indic_checker.dictionary, 1000)
        evaluate_indicbert_only(indic_checker, test_cases)
    except FileNotFoundError:
        print(f"Error: The file '{raw_file}' was not found. Please check your directory.")

 MODEL 8: IndicBERT-ONLY SPELL CHECKER
 IndicBERT-Only Initializing...
 Dictionary: 56,544 words
 Loading IndicBERT...


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
predictions.bias                 | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 
sop_classifier.classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 IndicBERT loaded: 768-dim embeddings
  IndicBERT Precomputing: 0/56544


model.safetensors:   0%|          | 0.00/135M [00:00<?, ?B/s]

  IndicBERT Precomputing: 1000/56544
  IndicBERT Precomputing: 2000/56544
  IndicBERT Precomputing: 3000/56544
  IndicBERT Precomputing: 4000/56544
  IndicBERT Precomputing: 5000/56544
  IndicBERT Precomputing: 6000/56544
  IndicBERT Precomputing: 7000/56544
  IndicBERT Precomputing: 8000/56544
  IndicBERT Precomputing: 9000/56544
  IndicBERT Precomputing: 10000/56544
  IndicBERT Precomputing: 11000/56544
  IndicBERT Precomputing: 12000/56544
  IndicBERT Precomputing: 13000/56544
  IndicBERT Precomputing: 14000/56544
  IndicBERT Precomputing: 15000/56544
  IndicBERT Precomputing: 16000/56544
  IndicBERT Precomputing: 17000/56544
  IndicBERT Precomputing: 18000/56544
  IndicBERT Precomputing: 19000/56544
  IndicBERT Precomputing: 20000/56544
  IndicBERT Precomputing: 21000/56544
  IndicBERT Precomputing: 22000/56544
  IndicBERT Precomputing: 23000/56544
  IndicBERT Precomputing: 24000/56544
  IndicBERT Precomputing: 25000/56544
  IndicBERT Precomputing: 26000/56544
  IndicBERT Precomput

#MuRIL

In [ ]:
import re
import random
import numpy as np
import torch
import time
from transformers import AutoModel, AutoTokenizer
from collections import defaultdict, Counter
import warnings

warnings.filterwarnings("ignore")

print("MuRIL SPELL CHECKER (Google Research)")
print("="*80)

#MuRIL EMBEDDING EXTRACTOR
class MuRILEmbedder:
    def __init__(self):
        print(" Loading MuRIL (google/muril-base-cased)...")
        self.model_name = "google/muril-base-cased"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModel.from_pretrained(self.model_name)
        self.model.eval()
        self.embedding_cache = {}
        print(" MuRIL loaded: 768-dim embeddings")

    def get_embedding(self, word):
        if word in self.embedding_cache:
            return self.embedding_cache[word]

        # Tokenize single word
        inputs = self.tokenizer(word, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = self.model(**inputs)
            # Use [CLS] token (index 0) as the word representation
            emb = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

        self.embedding_cache[word] = emb
        return emb

#MuRIL SPELL CHECKER
class MuRILSpellChecker:
    def __init__(self, raw_file_path):
        print(" MuRIL Initializing...")

        self.dictionary = self._load_dictionary(raw_file_path)
        print(f" Dictionary: {len(self.dictionary):,} words")

        self.embedder = MuRILEmbedder()
        self.word_embeddings = self._precompute_embeddings()
        self.length_index = self._build_length_index()
        print(" MuRIL Ready!")

    def _load_dictionary(self, file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()

        # Match Assamese Unicode range (0980-09FF)
        words = re.findall(r'[\u0980-\u09FF]{2,12}', text)
        return sorted(list(set(words)))

    def _precompute_embeddings(self):
        embeddings = {}
        total = len(self.dictionary)
        for i, word in enumerate(self.dictionary):
            if i % 1000 == 0:
                print(f"  MuRIL Precomputing: {i}/{total}")
            embeddings[word] = self.embedder.get_embedding(word)
        return embeddings

    def _build_length_index(self):
        index = defaultdict(list)
        for word in self.dictionary:
            index[len(word)].append(word)
        return dict(index)

    def get_embedding(self, word):
        return self.embedder.get_embedding(word)

    def muril_cosine_similarity(self, word1, word2):
        emb1 = self.get_embedding(word1)
        emb2 = self.word_embeddings[word2]
        dot = np.dot(emb1, emb2)
        norms = np.linalg.norm(emb1) * np.linalg.norm(emb2)
        return dot / norms if norms != 0 else 0.0

    def check(self, word):
        if len(word) < 2 or word in self.dictionary:
            return word in self.dictionary, []

        word_len = len(word)
        candidates = []
        # Length filtering
        for len_diff in [-1, 0, 1]:
            target_len = word_len + len_diff
            if target_len in self.length_index:
                candidates.extend(self.length_index[target_len][:200])

        candidates = list(set(candidates))[:500]
        scored = [(self.muril_cosine_similarity(word, cand), cand) for cand in candidates]
        top5 = sorted(scored, reverse=True)[:5]

        return False, top5

#EVALUATION
def generate_test_cases(dictionary, n_samples=200):
    error_swaps = {'া': 'ি', 'ি': 'া', 'ক': 'খ', 'খ': 'ক', 'গ': 'ঘ'}
    test_cases = []
    valid_words = [w for w in dictionary if 3 <= len(w) <= 10]

    for word in valid_words:
        for wrong, right in error_swaps.items():
            if wrong in word:
                misspelled = word.replace(wrong, right, 1)
                if misspelled != word and len(misspelled) >= 2:
                    test_cases.append({'original': word, 'misspelled': misspelled})
                    if len(test_cases) >= n_samples:
                        break
        if len(test_cases) >= n_samples:
            break
    print(f" Generated {len(test_cases)} test cases")
    return test_cases

def evaluate_muril_only(checker, test_cases):
    print("\n MuRIL RESULTS")
    print("="*60)

    top1_correct = top3_correct = 0
    start_time = time.time()

    print("Sample corrections:")
    for i, case in enumerate(test_cases[:10]):
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()

        top1_ok = case['original'] == top1_word
        status = 'OK' if case['original'] in top3_words else 'ERROR'
        print(f"  '{case['misspelled']}' -> {top1_word} | {case['original']} | {status}")

        top1_correct += int(top1_ok)
        top3_correct += int(case['original'] in top3_words)

    for case in test_cases[10:]:
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top1_correct += int(case['original'] == top1_word)
        if len(suggestions) >= 3:
            top3_words = {s[1] for s in suggestions[:3]}
            top3_correct += int(case['original'] in top3_words)

    total_duration = time.time() - start_time
    total = len(test_cases)

    print(f"\n{'='*60}")
    print(" MuRIL SPELL CHECKER")
    print(f" Test cases: {total}")
    print(f" Execution Time: {total_duration:.4f} seconds")
    print(f" Speed: {total_duration/total:.4f} sec/word")
    print(f" Top-1 Accuracy: {top1_correct/total:.1%}")
    print(f" Top-3 Accuracy: {top3_correct/total:.1%}")
    print(f"{'='*60}")

# Main
if __name__ == "__main__":
    raw_file = "News.txt"
    try:
        muril_checker = MuRILSpellChecker(raw_file)
        test_cases = generate_test_cases(muril_checker.dictionary, 1000)
        evaluate_muril_only(muril_checker, test_cases)
    except FileNotFoundError:
        print(f"Error: The file '{raw_file}' was not found.")

 MODEL 9: MuRIL-ONLY SPELL CHECKER (Google Research)
 MuRIL-Only Initializing...
 Dictionary: 56,544 words
 Loading MuRIL (google/muril-base-cased)...


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 MuRIL loaded: 768-dim embeddings
  MuRIL Precomputing: 0/56544
  MuRIL Precomputing: 1000/56544
  MuRIL Precomputing: 2000/56544
  MuRIL Precomputing: 3000/56544
  MuRIL Precomputing: 4000/56544
  MuRIL Precomputing: 5000/56544
  MuRIL Precomputing: 6000/56544
  MuRIL Precomputing: 7000/56544
  MuRIL Precomputing: 8000/56544
  MuRIL Precomputing: 9000/56544
  MuRIL Precomputing: 10000/56544
  MuRIL Precomputing: 11000/56544
  MuRIL Precomputing: 12000/56544
  MuRIL Precomputing: 13000/56544
  MuRIL Precomputing: 14000/56544
  MuRIL Precomputing: 15000/56544
  MuRIL Precomputing: 16000/56544
  MuRIL Precomputing: 17000/56544
  MuRIL Precomputing: 18000/56544
  MuRIL Precomputing: 19000/56544
  MuRIL Precomputing: 20000/56544
  MuRIL Precomputing: 21000/56544
  MuRIL Precomputing: 22000/56544
  MuRIL Precomputing: 23000/56544
  MuRIL Precomputing: 24000/56544
  MuRIL Precomputing: 25000/56544
  MuRIL Precomputing: 26000/56544
  MuRIL Precomputing: 27000/56544
  MuRIL Precomputing: 28000

#mBERT

In [ ]:
import re
import random
import numpy as np
import torch
import time
from transformers import AutoModel, AutoTokenizer
from collections import defaultdict, Counter
import warnings

warnings.filterwarnings("ignore")

print("mBERT SPELL CHECKER (Baseline Multilingual)")
print("="*80)

#mBERT EMBEDDING EXTRACTOR
class mBERTEmbedder:
    def __init__(self):
        print(" Loading mBERT (bert-base-multilingual-cased)...")
        self.model_name = "bert-base-multilingual-cased"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModel.from_pretrained(self.model_name)
        self.model.eval()
        self.embedding_cache = {}
        print(" mBERT loaded: 768-dim embeddings")

    def get_embedding(self, word):
        if word in self.embedding_cache:
            return self.embedding_cache[word]

        # Tokenize single word
        inputs = self.tokenizer(word, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = self.model(**inputs)
            # Use [CLS] token (index 0) for the word-level representation
            emb = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

        self.embedding_cache[word] = emb
        return emb

#mBERT SPELL CHECKER
class mBERTSpellChecker:
    def __init__(self, raw_file_path):
        print(" mBERT Initializing...")

        self.dictionary = self._load_dictionary(raw_file_path)
        print(f" Dictionary: {len(self.dictionary):,} words")

        self.embedder = mBERTEmbedder()
        self.word_embeddings = self._precompute_embeddings()
        self.length_index = self._build_length_index()
        print(" mBERT Ready!")

    def _load_dictionary(self, file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()

        # Match Assamese Unicode range (0980-09FF)
        words = re.findall(r'[\u0980-\u09FF]{2,12}', text)
        return sorted(list(set(words)))

    def _precompute_embeddings(self):
        embeddings = {}
        total = len(self.dictionary)
        for i, word in enumerate(self.dictionary):
            if i % 1000 == 0:
                print(f"  mBERT Precomputing: {i}/{total}")
            embeddings[word] = self.embedder.get_embedding(word)
        return embeddings

    def _build_length_index(self):
        index = defaultdict(list)
        for word in self.dictionary:
            index[len(word)].append(word)
        return dict(index)

    def get_embedding(self, word):
        return self.embedder.get_embedding(word)

    def mbert_cosine_similarity(self, word1, word2):
        emb1 = self.get_embedding(word1)
        emb2 = self.word_embeddings[word2]
        dot = np.dot(emb1, emb2)
        norms = np.linalg.norm(emb1) * np.linalg.norm(emb2)
        return dot / norms if norms != 0 else 0.0

    def check(self, word):
        if len(word) < 2 or word in self.dictionary:
            return word in self.dictionary, []

        word_len = len(word)
        candidates = []
        # Consistent length filtering (±1 char)
        for len_diff in [-1, 0, 1]:
            target_len = word_len + len_diff
            if target_len in self.length_index:
                candidates.extend(self.length_index[target_len][:200])

        candidates = list(set(candidates))[:500]
        scored = [(self.mbert_cosine_similarity(word, cand), cand) for cand in candidates]
        top5 = sorted(scored, reverse=True)[:5]

        return False, top5

# EVALUATION
def generate_test_cases(dictionary, n_samples=200):
    error_swaps = {'া': 'ি', 'ি': 'া', 'ক': 'খ', 'খ': 'ক', 'গ': 'ঘ'}
    test_cases = []
    valid_words = [w for w in dictionary if 3 <= len(w) <= 10]

    for word in valid_words:
        for wrong, right in error_swaps.items():
            if wrong in word:
                misspelled = word.replace(wrong, right, 1)
                if misspelled != word and len(misspelled) >= 2:
                    test_cases.append({'original': word, 'misspelled': misspelled})
                    if len(test_cases) >= n_samples:
                        break
        if len(test_cases) >= n_samples:
            break
    print(f" Generated {len(test_cases)} test cases")
    return test_cases

def evaluate_mbert_only(checker, test_cases):
    print("\n mBERT RESULTS")
    print("="*60)

    top1_correct = top3_correct = 0
    start_time = time.time()

    print("Sample corrections:")
    for i, case in enumerate(test_cases[:10]):
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top3_words = {s[1] for s in suggestions[:3]} if len(suggestions) >= 3 else set()

        top1_ok = case['original'] == top1_word
        status = 'OK ' if case['original'] in top3_words else 'ERROR '
        print(f"  '{case['misspelled']}' -> {top1_word} | {case['original']} | {status}")

        top1_correct += int(top1_ok)
        top3_correct += int(case['original'] in top3_words)

    for case in test_cases[10:]:
        is_correct, suggestions = checker.check(case['misspelled'])
        top1_word = suggestions[0][1] if suggestions else ""
        top1_correct += int(case['original'] == top1_word)
        if len(suggestions) >= 3:
            top3_words = {s[1] for s in suggestions[:3]}
            top3_correct += int(case['original'] in top3_words)

    total_duration = time.time() - start_time
    total = len(test_cases)

    print(f"\n{'='*60}")
    print(" mBERT SPELL CHECKER")
    print(f" Test cases: {total}")
    print(f" Execution Time: {total_duration:.4f} seconds")
    print(f" Speed: {total_duration/total:.4f} sec/word")
    print(f" Top-1 Accuracy: {top1_correct/total:.1%}")
    print(f" Top-3 Accuracy: {top3_correct/total:.1%}")
    print(f"{'='*60}")

#Main
if __name__ == "__main__":
    raw_file = "News.txt"
    try:
        mbert_checker = mBERTSpellChecker(raw_file)
        test_cases = generate_test_cases(mbert_checker.dictionary, 1000)
        evaluate_mbert_only(mbert_checker, test_cases)
    except FileNotFoundError:
        print(f"Error: The file '{raw_file}' was not found.")

 MODEL 10: mBERT-ONLY SPELL CHECKER (Baseline Multilingual)
 mBERT-Only Initializing...
 Dictionary: 56,544 words
 Loading mBERT (bert-base-multilingual-cased)...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 mBERT loaded: 768-dim embeddings
  mBERT Precomputing: 0/56544
  mBERT Precomputing: 1000/56544
  mBERT Precomputing: 2000/56544
  mBERT Precomputing: 3000/56544
  mBERT Precomputing: 4000/56544
  mBERT Precomputing: 5000/56544
  mBERT Precomputing: 6000/56544
  mBERT Precomputing: 7000/56544
  mBERT Precomputing: 8000/56544
  mBERT Precomputing: 9000/56544
  mBERT Precomputing: 10000/56544
  mBERT Precomputing: 11000/56544
  mBERT Precomputing: 12000/56544
  mBERT Precomputing: 13000/56544
  mBERT Precomputing: 14000/56544
  mBERT Precomputing: 15000/56544
  mBERT Precomputing: 16000/56544
  mBERT Precomputing: 17000/56544
  mBERT Precomputing: 18000/56544
  mBERT Precomputing: 19000/56544
  mBERT Precomputing: 20000/56544
  mBERT Precomputing: 21000/56544
  mBERT Precomputing: 22000/56544
  mBERT Precomputing: 23000/56544
  mBERT Precomputing: 24000/56544
  mBERT Precomputing: 25000/56544
  mBERT Precomputing: 26000/56544
  mBERT Precomputing: 27000/56544
  mBERT Precomputing: 28000